# Crafting an AI-Powered HR Assistant
## A retrieval-augmented chatbot for *The Nestlé Human Resources Policy*

**Course-End Project · Conversational AI with OpenAI GPT, LangChain, Chroma DB and Gradio**

### Goal
Build a conversational chatbot that answers employees' questions about Nestlé's HR policy **accurately** — by retrieving the
relevant passages from the policy PDF and letting GPT-3.5 Turbo compose an answer grounded in those passages — all behind a
user-friendly Gradio chat interface.

### Architecture — Retrieval-Augmented Generation (RAG)

```
 nestle_hr_policy.pdf ──PyPDFLoader──► pages ──RecursiveCharacterTextSplitter──► chunks
                                                                                   │
                                                             OpenAI embeddings ──► Chroma vector store (persisted)
                                                                                   │
 user question ──embed──► top-k similar chunks ──► prompt template ──► GPT-3.5 Turbo ──► grounded answer + page citations
                                                                                                        │
                                                                                            Gradio ChatInterface
```

### Notebook map
| # | Section | Requirement from the brief |
|---|---------|----------------------------|
| 1 | Environment setup & imports | Import essential tools |
| 2 | OpenAI API environment | Set up OpenAI's API environment |
| 3 | Load the PDF | Load Nestlé's HR policy using PyPDFLoader |
| 4 | Split into chunks | …and split it for easy processing |
| 5 | Embeddings + Chroma DB | Vector representations using Chroma DB and OpenAI embeddings |
| 6 | GPT-3.5 Turbo + retriever | Question-answering system retrieving answers from text chunks |
| 7 | Prompt template | Guide the chatbot in understanding and responding to users |
| 8 | Tests | Verify answers, sources and off-topic handling |
| 9 | Gradio chatbot UI | User-friendly interface for interaction and information retrieval |
| 10 | Conclusion | Results, limitations and next steps |

> **How to run:** make sure `data/nestle_hr_policy.pdf` exists (see `README.md`), have your `OPENAI_API_KEY` ready
> (environment variable, `.env` file, or type it when prompted), then run all cells top-to-bottom.
> Works locally (Jupyter / VS Code) and on Google Colab.


## 1. Environment setup

The exact, tested package versions are pinned so the notebook runs the same way everywhere.
Re-running the cell is safe — packages already present are skipped.


In [ ]:
# Install the pinned dependencies (quiet). Skip this cell if you already ran `pip install -r requirements.txt`.
%pip install -q "langchain-core==1.6.6" "langchain-community==0.4.2" "langchain-text-splitters==1.1.3" "langchain-openai==1.6.7" "langchain-chroma==1.1.0" "chromadb==1.5.9" "pypdf==6.19.0" "openai==3.24.0" "gradio==6.29.1" "python-dotenv==1.2.4"

In [ ]:
import os
import getpass
import textwrap
import warnings
from pathlib import Path
from importlib.metadata import version as pkg_version

from dotenv import load_dotenv

# Chroma phones home with anonymous usage stats by default; keep the project self-contained.
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
# langchain-community is being sunset in favour of standalone packages, but it is still where PyPDFLoader (required
# by the brief) lives and the pinned version works fine — silence its import-time notice.
warnings.filterwarnings("ignore", message=".*langchain-community.*")

# LangChain building blocks
from langchain_community.document_loaders import PyPDFLoader          # PDF -> Documents
from langchain_text_splitters import RecursiveCharacterTextSplitter    # Documents -> chunks
from langchain_openai import OpenAIEmbeddings, ChatOpenAI              # OpenAI embeddings + GPT-3.5 Turbo
from langchain_chroma import Chroma                                    # vector store
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import gradio as gr                                                    # chatbot UI
from openai import OpenAI                                              # used once, to check the model is available

for pkg in ["langchain-core", "langchain-community", "langchain-openai", "langchain-chroma",
            "chromadb", "pypdf", "openai", "gradio"]:
    print(f"{pkg:<20} {pkg_version(pkg)}")

## 2. OpenAI API environment

The key is read from the `OPENAI_API_KEY` environment variable (a local `.env` file is loaded automatically — see
`.env.example`). If it is not set, you are prompted for it securely: the key is never printed or saved in the notebook.


In [ ]:
load_dotenv()  # picks up OPENAI_API_KEY from a .env file, if there is one

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("OpenAI API key loaded:", "yes" if os.getenv("OPENAI_API_KEY") else "no")

## 3. Load Nestlé's HR policy with PyPDFLoader

`PyPDFLoader` turns the PDF into one LangChain `Document` per page. Each document carries the page text plus metadata
(`source`, `page`) — we keep the page number all the way through so every answer can cite where it came from.


In [ ]:
PDF_PATH = Path("data/nestle_hr_policy.pdf")   # "The Nestlé Human Resources Policy" (public document)
CHROMA_DIR = "chroma_db"                        # where the vector store is persisted on disk

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-3.5-turbo")   # required by the brief; "gpt-4o-mini" is a drop-in upgrade

COLLECTION_NAME = f"nestle_hr_policy__{EMBEDDING_MODEL}"   # one collection per embedding model -> never re-use stale vectors

CHUNK_SIZE = 1000       # characters per chunk — large enough for a full policy paragraph
CHUNK_OVERLAP = 150     # overlap keeps sentences that straddle a boundary intact
TOP_K = 4               # chunks retrieved per question

assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH.resolve()} — see README.md for how to obtain it."

In [ ]:
loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

total_chars = sum(len(p.page_content) for p in pages)
print(f"Loaded {len(pages)} pages, {total_chars:,} characters in total")
print("Metadata example:", {k: pages[0].metadata[k] for k in ("source", "page") if k in pages[0].metadata})

richest_page = max(pages, key=lambda p: len(p.page_content))   # the page with the most text (skips cover/imprint)
print(f"\nPreview of page {richest_page.metadata['page'] + 1} (the most text-rich page):\n" + "-" * 70)
print(textwrap.shorten(richest_page.page_content, width=700, placeholder=" …"))

## 4. Split the text into chunks

Whole pages are too coarse to retrieve precisely and too long to stuff many of them into a prompt.
`RecursiveCharacterTextSplitter` cuts the text into ~1 000-character chunks, trying paragraph breaks first, then line
breaks, then sentences, so chunks stay semantically coherent. A 150-character overlap prevents losing a sentence that
sits exactly on a boundary.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],   # paragraph -> line -> sentence -> word
)
chunks = splitter.split_documents(pages)

lengths = [len(c.page_content) for c in chunks]
print(f"{len(pages)} pages -> {len(chunks)} chunks")
print(f"chunk length: min {min(lengths)}, avg {sum(lengths) // len(lengths)}, max {max(lengths)} characters")

sample = chunks[len(chunks) // 2]
print(f"\nSample chunk (page {sample.metadata.get('page', 0) + 1}):\n" + "-" * 70)
print(sample.page_content)

## 5. Vector representations with OpenAI embeddings + Chroma DB

Each chunk is converted into a numerical vector with OpenAI's embedding model and stored in a **Chroma** collection.
Semantically similar text ends up close together in vector space, which is what lets us find the passages relevant to a
question.

The collection is **persisted** to `chroma_db/`, so re-running the notebook re-uses the index instead of paying for the
embeddings again (a fresh collection is built automatically if the chunking or the embedding model changes).


In [ ]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

existing = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
n_existing = len(existing.get()["ids"])

if n_existing == len(chunks):
    vectorstore = existing   # complete index already on disk -> reuse it
    print(f"Loaded existing Chroma collection with {n_existing} vectors from '{CHROMA_DIR}/'")
else:
    if n_existing:
        existing.delete_collection()   # stale/partial index -> rebuild from scratch
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DIR,
    )
    print(f"Embedded {len(chunks)} chunks with '{EMBEDDING_MODEL}' and persisted them to '{CHROMA_DIR}/'")

**Sanity check** — a raw similarity search (no LLM involved yet). Lower distance = more similar.

In [ ]:
query = "How does Nestlé support training and learning for employees?"

for rank, (doc, distance) in enumerate(vectorstore.similarity_search_with_score(query, k=3), start=1):
    print(f"#{rank}  distance={distance:.3f}  page={doc.metadata.get('page', 0) + 1}")
    print(textwrap.shorten(doc.page_content, width=300, placeholder=" …"), "\n")

## 6. Question-answering with GPT-3.5 Turbo

The vector store becomes a **retriever** that returns the `TOP_K` most relevant chunks for a question, and
`ChatOpenAI` wraps GPT-3.5 Turbo. `temperature=0` makes the answers factual and repeatable — exactly what you want
from an HR assistant. Should OpenAI have retired the configured model for your account, the cell falls back to
`gpt-4o-mini` and prints a note.


In [ ]:
try:
    OpenAI().models.retrieve(LLM_MODEL)            # cheap metadata call — no tokens used
except Exception as exc:                           # e.g. a GPT-3.5 snapshot that OpenAI has retired
    print(f"'{LLM_MODEL}' is not available for this API key ({type(exc).__name__}); using 'gpt-4o-mini' instead")
    LLM_MODEL = "gpt-4o-mini"

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K},
)
print(f"LLM: {LLM_MODEL}  |  retriever returns the top-{TOP_K} chunks per question")

## 7. Prompt template

The prompt is what turns a general-purpose model into a trustworthy HR assistant. It

* fixes the **persona** and tone,
* restricts answers to the **retrieved context** (no hallucinated policies),
* defines an explicit **fallback** when the document does not cover a question,
* and includes the **conversation history**, so follow-up questions work.

The pieces are wired together with the LangChain Expression Language (LCEL). The chain returns both the answer and
the source chunks, so the UI can cite pages.


In [ ]:
NOT_FOUND_MESSAGE = "I couldn't find that in the Nestlé HR Policy document."

SYSTEM_PROMPT = f"""You are Nestlé's HR Assistant, a helpful and precise chatbot for employees and HR staff.
You answer questions using ONLY the excerpts provided from "The Nestlé Human Resources Policy".

Guidelines:
1. Ground every statement in the provided context. Never invent policies, numbers or entitlements.
2. If the context does not contain the answer, reply exactly: "{NOT_FOUND_MESSAGE}" and suggest contacting the local HR team.
3. Be concise and well structured — short paragraphs or bullet points.
4. When helpful, name the policy area the answer comes from (e.g. "Training and learning", "Total rewards").
5. Use the conversation history to understand follow-up questions, but still answer from the context.
6. Keep a professional, friendly tone."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "Context from the HR policy:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    """Render retrieved chunks as numbered, page-tagged context blocks."""
    return "\n\n".join(
        f"[Excerpt {i} — page {d.metadata.get('page', 0) + 1}]\n{d.page_content}"
        for i, d in enumerate(docs, start=1)
    )


# The RAG chain:
#   {question, chat_history} --retrieve--> + docs --format--> + context --prompt--> GPT-3.5 Turbo --> answer
retrieve = RunnablePassthrough.assign(docs=lambda x: retriever.invoke(x["question"]))
generate = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain = retrieve | RunnablePassthrough.assign(answer=generate)


def ask(question, chat_history=None):
    """Answer a question; returns (answer, source_documents)."""
    result = rag_chain.invoke({"question": question, "chat_history": chat_history or []})
    return result["answer"], result["docs"]


print(prompt.format(context="<retrieved excerpts>", question="<user question>", chat_history=[]))

## 8. Test the assistant

A few representative HR questions, plus one **off-topic** question to confirm the assistant declines gracefully instead
of making something up.


In [ ]:
sample_questions = [
    "What is the purpose of the Nestlé Human Resources Policy?",
    "How does Nestlé approach training and learning for its employees?",
    "What does the policy say about work-life balance and flexible working?",
    "How does Nestlé handle remuneration and total rewards?",
    "What is the capital of France?",   # off-topic -> should answer with NOT_FOUND_MESSAGE
]

for q in sample_questions:
    answer, docs = ask(q)
    pages_used = sorted({d.metadata.get("page", 0) + 1 for d in docs})
    print("Q:", q)
    print("A:", textwrap.fill(answer, width=100, subsequent_indent="   "))
    print("   sources: pages", pages_used)
    print("=" * 100)

**Follow-up questions** — the chat history is passed to the prompt, so the assistant understands references such as
"that" or "it".

In [ ]:
history = []

q1 = "Does the policy mention flexible working arrangements?"
a1, _ = ask(q1, history)
history += [HumanMessage(q1), AIMessage(a1)]
print("Q1:", q1, "\nA1:", a1, "\n")

q2 = "Summarise that in two short bullet points."   # only makes sense with the history
a2, _ = ask(q2, history)
print("Q2:", q2, "\nA2:", a2)

## 9. Gradio chatbot interface

`gr.ChatInterface` gives us a complete chat UI from a single function `respond(message, history)`. The callback converts
Gradio's history into LangChain messages, runs the RAG chain, and appends the pages the answer was drawn from.

* The UI renders inline in Jupyter/Colab and is also served at the printed local URL.
* Set `share=True` to get a temporary public link (useful to demo from Colab).


In [ ]:
def history_to_messages(history, max_turns=6):
    """Convert Gradio chat history into LangChain messages (keeps the last `max_turns` exchanges)."""
    messages = []
    for item in history:
        if isinstance(item, dict):                      # Gradio "messages" format
            content = item.get("content")
            if isinstance(content, list):               # multimodal payloads -> keep the text parts
                content = " ".join(part.get("text", "") for part in content if isinstance(part, dict))
            if item.get("role") == "user":
                messages.append(HumanMessage(str(content)))
            elif item.get("role") == "assistant":
                messages.append(AIMessage(str(content)))
        else:                                           # legacy (user, assistant) tuples
            user_msg, bot_msg = item
            messages += [HumanMessage(str(user_msg)), AIMessage(str(bot_msg or ""))]
    return messages[-max_turns * 2:]


def respond(message, history):
    """Gradio callback: answer the user's message and cite the pages it came from."""
    answer, docs = ask(message, history_to_messages(history))
    if NOT_FOUND_MESSAGE.lower() not in answer.lower():
        pages_used = ", ".join(str(p) for p in sorted({d.metadata.get("page", 0) + 1 for d in docs}))
        answer += f"\n\n📄 *Source: The Nestlé Human Resources Policy — page(s) {pages_used}*"
    return answer


demo = gr.ChatInterface(
    fn=respond,
    title="Nestlé HR Assistant",
    description=(
        "Ask anything about **The Nestlé Human Resources Policy**. "
        "Answers are generated by GPT-3.5 Turbo from the policy text and cite the pages used."
    ),
    examples=[
        "What are the key principles of Nestlé's HR policy?",
        "How does Nestlé support employee training and development?",
        "What does the policy say about diversity and inclusion?",
        "How is employee performance evaluated and rewarded?",
    ],
)

demo.launch(share=False)   # share=True -> temporary public link

## 10. Conclusion

### What was built
| Requirement | Implementation |
|-------------|----------------|
| Import tools & set up the OpenAI environment | LangChain, Chroma, OpenAI and Gradio imports; key loaded from env / `.env` / secure prompt |
| Load the HR policy with PyPDFLoader & split it | `PyPDFLoader` → page documents → `RecursiveCharacterTextSplitter` (1 000 chars, 150 overlap) |
| Vector representations with Chroma DB & OpenAI embeddings | `text-embedding-3-small` vectors in a persisted Chroma collection |
| Question-answering with GPT-3.5 Turbo | Retriever (top-4 chunks) + `ChatOpenAI(gpt-3.5-turbo, temperature=0)` in an LCEL chain |
| Prompt template | Persona, grounding rules, explicit fallback, conversation history, page citations |
| Gradio chatbot UI | `gr.ChatInterface` with examples, history-aware answers and source pages |

### Why it works well for HR
* **Accuracy** – answers are grounded in retrieved policy text and cite pages, so employees can verify them.
* **Safety** – the prompt forbids inventing entitlements and gives a clear fallback for uncovered topics.
* **Efficiency** – the Chroma index is persisted; only the question is embedded at query time.
* **Accessibility** – a chat UI anyone can use, deployable with `python app.py` or on Hugging Face Spaces.

### Limitations & next steps
* Follow-up questions are retrieved with the raw question; adding a *query-rewriting* step (condense the question with
  the history before retrieval) would improve multi-turn accuracy.
* Add an evaluation set of HR questions with expected answers to measure retrieval and answer quality over time.
* Hybrid search (BM25 + vectors) or a re-ranker would help with very specific keyword questions.
* For production: authentication, logging/monitoring, and keeping the index in sync with policy updates.
